# 🏦 Robust NLP Framework for Automated Financial Transaction Extraction and Expense Categorization

**School of Computer Science and Engineering — RV University**  
**Team ID: 10**  
- **Preksha P V** (`1RUA24CSE0331`)
- **Reshma K P** (`1RUA24CSE0363`)
- **Prathik K B** (`1RUA24CSE0327`)
- **Rushil Ravi Reddy** (`1RUA24CSE0380`)

**Guides:** Prof. Khushboo Pandey & Prof. Priyanka S Shivaramaiah  
**Dataset:** [`Ranjit0034/finee-dataset`](https://huggingface.co/datasets/Ranjit0034/finee-dataset) (152,000+ Indian Banking & UPI Messages across 6 Indian Languages + Code-Mixed Hinglish/Kanglish/Tanglish)

---
### ⚡ Workflow: Train on Colab GPU → Export & Run Locally on MacBook
1. **Steps 1–7:** Train the Subword SVM, LightGBM, and fine-tune **MuRIL (`google/muril-base-cased`)** on Colab GPU in ~2 minutes.
2. **Step 8:** Export and download `trained_models.zip` to your MacBook so you can run instant local inference without retraining.

## Step 1: Install Required Packages & Verify Colab Pro GPU

In [ ]:
!pip install -q datasets transformers accelerate scikit-learn lightgbm plotly gradio seaborn fastapi uvicorn pycloudflared nest_asyncio

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device     :", torch.cuda.get_device_name(0))

## Step 2: Multilingual & Code-Mixed Financial Lexicons and Preprocessor
Supports **English (`en`)**, **Hindi (`hi`)**, **Kannada (`kn`)**, **Tamil (`ta`)**, **Telugu (`te`)**, **Bengali (`bn`)**, and **Romanized Code-Mixed (`hinglish` / `kanglish` / `tanglish`)** banking messages.

In [ ]:
import re
import json
import random
from typing import Dict, List, Tuple, Optional, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SUPPORTED_LANGUAGES = {
    "en": "English",
    "hi": "Hindi (हिंदी)",
    "ta": "Tamil (தமிழ்)",
    "te": "Telugu (తెలుగు)",
    "bn": "Bengali (বাংলা)",
    "kn": "Kannada (ಕನ್ನಡ)",
    "hinglish": "Code-Mixed (Hinglish / Kanglish / Tanglish)",
}

SYNOPSIS_CATEGORY_MAP = {
    "food": "Food & Dining",
    "grocery": "Food & Dining",
    "shopping": "Shopping",
    "transport": "Travel & Transport",
    "travel": "Travel & Transport",
    "bills": "Bills & Utilities",
    "entertainment": "Entertainment & Lifestyle",
    "healthcare": "Healthcare",
    "education": "Education",
    "investment": "Investments & Wealth",
    "emi": "Bills & Utilities",
    "transfer": "P2P Transfers",
    "salary": "Income & Salary",
}

INDIAN_BANKS = {
    "HDFC": ["hdfc", "hdfcbank", "एचडीएफसी", "హెచ్‌డిఎఫ్‌సి", "எச்டிஎஃப்சி", "এইচডিএফসি", "ಎಚ್‌ಡಿಎಫ್‌ಸಿ"],
    "SBI": ["sbi", "state bank of india", "state bank", "एसबीआई", "ఎస్‌బిఐ", "எஸ்பிஐ", "এসবিআই", "ಎಸ್‌ಬಿಐ"],
    "ICICI": ["icici", "icicibank", "आईसीआईसीआई", "ఐసిఐసిఐ", "ஐசிஐசிஐ", "আইসিআইসিআই", "ಐಸಿಐಸಿಐ"],
    "Axis": ["axis", "axis bank", "axisbank", "एक्सिस", "యాక్సిస్", "ஆக்சிஸ்", "অ্যাক্সিস", "ಆಕ್ಸಿಸ್"],
    "Kotak": ["kotak", "kotak mahindra", "कोटक", "కోటక్", "கோடக்", "কোটাক", "ಕೋಟಕ್"],
    "PNB": ["pnb", "punjab national bank", "पीएनबी"],
    "BOB": ["bob", "bank of baroda", "baroda", "बैंक ऑफ बड़ौदा"],
    "Canara": ["canara", "canara bank", "केनरा", "ಕೆನರಾ"],
    "Union": ["union bank", "union"],
    "IDBI": ["idbi", "idbi bank"],
    "IndusInd": ["indusind", "indusind bank"],
    "YES Bank": ["yes bank", "yesbank"],
    "Federal": ["federal", "federal bank"],
    "IDFC First": ["idfc", "idfc first"],
    "Amex": ["amex", "american express"],
}

MERCHANT_LEXICON = {
    # Food & Delivery
    "swiggy": ("Swiggy", "food"), "zomato": ("Zomato", "food"), "dominos": ("Domino's", "food"),
    "mcdonalds": ("McDonald's", "food"), "kfc": ("KFC", "food"), "starbucks": ("Starbucks", "food"),
    "burger king": ("Burger King", "food"), "pizza hut": ("Pizza Hut", "food"), "eatfit": ("EatFit", "food"),
    # Grocery
    "zepto": ("Zepto", "grocery"), "bigbasket": ("BigBasket", "grocery"), "blinkit": ("Blinkit", "grocery"),
    "instamart": ("Instamart", "grocery"), "dmart": ("DMart", "grocery"), "jiomart": ("JioMart", "grocery"),
    # Shopping
    "amazon": ("Amazon", "shopping"), "flipkart": ("Flipkart", "shopping"), "myntra": ("Myntra", "shopping"),
    "meesho": ("Meesho", "shopping"), "ajio": ("AJIO", "shopping"), "nykaa": ("Nykaa", "shopping"),
    "tatacliq": ("Tata CLiQ", "shopping"), "croma": ("Croma", "shopping"), "decathlon": ("Decathlon", "shopping"),
    "zudio": ("Zudio", "shopping"), "westside": ("Westside", "shopping"), "lenskart": ("Lenskart", "shopping"),
    # Transport & Travel
    "uber": ("Uber", "transport"), "ola": ("Ola", "transport"), "rapido": ("Rapido", "transport"),
    "namma yatri": ("Namma Yatri", "transport"), "indian oil": ("Indian Oil", "transport"), "hpcl": ("HPCL", "transport"),
    "bpcl": ("BPCL", "transport"), "shell": ("Shell", "transport"), "fastag": ("FASTag", "transport"),
    "irctc": ("IRCTC", "travel"), "makemytrip": ("MakeMyTrip", "travel"), "goibibo": ("Goibibo", "travel"),
    "redbus": ("redBus", "travel"), "indigo": ("IndiGo", "travel"), "air india": ("Air India", "travel"),
    "vistara": ("Vistara", "travel"), "oyo": ("OYO", "travel"), "airbnb": ("Airbnb", "travel"),
    # Bills & Utilities
    "airtel": ("Airtel", "bills"), "jio": ("Jio", "bills"), "vodafone": ("Vi", "bills"), "bsnl": ("BSNL", "bills"),
    "act fibernet": ("ACT Fibernet", "bills"), "bescom": ("BESCOM", "bills"), "tata power": ("Tata Power", "bills"),
    "cesc": ("CESC", "bills"), "indane": ("Indane Gas", "bills"), "hp gas": ("HP Gas", "bills"),
    "tata play": ("Tata Play", "bills"), "cred": ("CRED", "bills"),
    # Entertainment
    "netflix": ("Netflix", "entertainment"), "bookmyshow": ("BookMyShow", "entertainment"),
    "hotstar": ("Hotstar", "entertainment"), "jiocinema": ("JioCinema", "entertainment"),
    "spotify": ("Spotify", "entertainment"), "prime video": ("Amazon Prime", "entertainment"),
    "sonyliv": ("SonyLIV", "entertainment"), "pvr": ("PVR Inox", "entertainment"),
    # Healthcare
    "apollo": ("Apollo", "healthcare"), "pharmeasy": ("PharmEasy", "healthcare"), "1mg": ("Tata 1mg", "healthcare"),
    "netmeds": ("Netmeds", "healthcare"), "medplus": ("MedPlus", "healthcare"), "practo": ("Practo", "healthcare"),
    "manipal": ("Manipal Hospital", "healthcare"), "fortis": ("Fortis", "healthcare"), "cult.fit": ("Cult.fit", "healthcare"),
    "cultfit": ("Cult.fit", "healthcare"),
    # Education
    "udemy": ("Udemy", "education"), "coursera": ("Coursera", "education"), "unacademy": ("Unacademy", "education"),
    "byjus": ("BYJU'S", "education"), "physicswallah": ("PhysicsWallah", "education"), "pw@": ("PhysicsWallah", "education"),
    "upgrad": ("upGrad", "education"), "scaler": ("Scaler Academy", "education"), "rv university": ("RV University", "education"),
    # Investment
    "zerodha": ("Zerodha", "investment"), "groww": ("Groww", "investment"), "upstox": ("Upstox", "investment"),
    "angel one": ("Angel One", "investment"), "icici direct": ("ICICI Direct", "investment"),
    "icicidirect": ("ICICI Direct", "investment"), "dhan": ("Dhan", "investment"), "paytm money": ("Paytm Money", "investment"),
    "paytmmoney": ("Paytm Money", "investment"), "kuvera": ("Kuvera", "investment"), "lic": ("LIC", "investment"),
}

SCRIPT_RANGES = {
    "hi": (0x0900, 0x097F),
    "bn": (0x0980, 0x09FF),
    "ta": (0x0B80, 0x0BFF),
    "te": (0x0C00, 0x0C7F),
    "kn": (0x0C80, 0x0CFF),
}

CODE_MIXED_LEXICON = {
    "mein": "in", "se": "from", "ko": "to", "ka": "of", "hua": "happened", "gaya": "went",
    "aaya": "came", "mila": "received", "bheja": "sent", "khata": "account", "khate": "account",
    "paisa": "money", "rupaye": "rupees", "jama": "credited", "kat": "debited", "bhugtan": "payment",
    "nimma": "your", "inda": "from", "agide": "happened", "mulaka": "via",
    "unga": "your", "irundhu": "from", "aagiduchu": "happened",
}

INDIC_FINANCIAL_GLOSSARY = {
    "खाता": "account", "खाते": "account", "डेबिट": "debited", "क्रेडिट": "credited", "दिनांक": "date", "संदर्भ": "reference",
    "ఖాతా": "account", "నుండి": "from", "డెబిట్": "debited", "క్రెడిట్": "credited", "తేదీ": "date",
    "கணக்கு": "account", "டெபிட்": "debited", "கிரெடிட்": "credited", "தேதி": "date",
    "অ্যাকাউন্ট": "account", "ডেবিট": "debited", "ক্রেডিট": "credited", "তারিখ": "date",
    "ಖಾತೆ": "account", "ಡೆಬಿಟ್": "debited", "ಕ್ರೆಡಿಟ್": "credited", "ದಿನಾಂಕ": "date",
}

PROMPT_PREFIX_REGEX = re.compile(
    r"^(?:Extract the financial entities from the following message:|"
    r"Extract structured data from this bank notification:|"
    r"Identify transaction details in this message:|"
    r"Analyze this transaction message and extract details:|"
    r"Identify and extract all transaction details:|"
    r"Parse this banking SMS and extract entities:|"
    r"Parse the financial information from this banking message:|"
    r"Extract financial entities from this message:|"
    r"Extract financial entities from:)\s*",
    re.IGNORECASE,
)

def strip_instruction_prompt(text: str) -> str:
    return PROMPT_PREFIX_REGEX.sub("", str(text).strip()).strip()

def detect_language_and_codemixing(text: str) -> Tuple[str, bool]:
    clean = strip_instruction_prompt(text)
    counts = {k: 0 for k in SCRIPT_RANGES}
    latin = 0
    for ch in clean:
        cp = ord(ch)
        if (0x0041 <= cp <= 0x005A) or (0x0061 <= cp <= 0x007A):
            latin += 1
        else:
            for lang, (lo, hi) in SCRIPT_RANGES.items():
                if lo <= cp <= hi:
                    counts[lang] += 1
                    break
    total = latin + sum(counts.values())
    if total == 0:
        return "en", False
    top_lang, top_cnt = max(counts.items(), key=lambda x: x[1])
    if top_cnt > 0:
        is_cm = (latin / total) >= 0.15 and (top_cnt / total) >= 0.10
        return top_lang, is_cm
    tokens = re.findall(r"[a-zA-Z]+", clean.lower())
    if any(t in CODE_MIXED_LEXICON for t in tokens):
        return "hinglish", True
    return "en", False

def normalize_financial_text(text: str) -> str:
    raw = strip_instruction_prompt(text)
    norm = re.sub(r"(?:₹|INR|Rs\.?|रु\.?|రూ\.?|ரூ\.?|৳|ರೂ\.?)\s*", "Rs. ", raw, flags=re.IGNORECASE)
    aug = []
    for w in norm.split():
        cw = w.strip(".,;:!()[]{}\"'।|").lower()
        aug.append(w)
        if cw in INDIC_FINANCIAL_GLOSSARY:
            aug.append(INDIC_FINANCIAL_GLOSSARY[cw])
        elif cw in CODE_MIXED_LEXICON:
            aug.append(CODE_MIXED_LEXICON[cw])
    return re.sub(r"\s+", " ", " ".join(aug)).strip()

print("Preprocessor & Multilingual Glossaries initialized.")

## Step 3: Hybrid Financial Entity Extractor (Rule-Based Regex + Contextual NER)

In [ ]:
DEBIT_KEYWORDS = [
    "debited", "debit", "spent", "paid", "charged", "sent", "withdrawn", "deducted", "emi of", "due",
    "डेबिट", "काटे", "भुगतान", "kat gaya", "debit hua", "ಡెబిట్", "డెబిట్", "டெபிட்", "ডেবিট", "ಡೆಬಿಟ್", "debit agide", "debit aagiduchu"
]
CREDIT_KEYWORDS = [
    "credited", "credit", "received", "deposited", "refund", "cashback", "salary", "added to", "neft from",
    "क्रेडिट", "जमा", "credit hua", "jama hua", "క్రెడిట్", "கிரெடிட்", "ক্রেডিট", "ಕ್ರೆಡಿಟ್"
]

AMOUNT_PATTERNS = [
    re.compile(r"(?:Rs\.?|INR|₹|रु\.?|రూ\.?|ரூ\.?|৳|ರೂ\.?)\s*([0-9]{1,3}(?:,[0-9]{2,3})+(?:\.[0-9]{1,2})?|[0-9]+(?:\.[0-9]{1,2})?)", re.IGNORECASE),
    re.compile(r"\b([0-9]{1,3}(?:,[0-9]{2,3})+(?:\.[0-9]{1,2})?|[0-9]+(?:\.[0-9]{1,2})?)\s*(?:rs\.?|inr|rupees|rupaye)\b", re.IGNORECASE),
]
ACCOUNT_PATTERNS = [
    re.compile(r"(?:A/c|Acct|Account|Card|CC|Loan|खाता|खाते|ఖాతా|கணக்கு|অ্যাকাউন্ট|ಖಾತೆ|khate|ending)\s*(?:no\.?|number)?\s*(?:[xX\*]+)?([0-9]{3,4})\b", re.IGNORECASE),
    re.compile(r"\b[xX\*]{2,6}([0-9]{3,4})\b"),
]
DATE_PATTERNS = [
    re.compile(r"\b([0-3][0-9][-/.][0-1][0-9][-/.](?:20)?[2-9][0-9])\b"),
    re.compile(r"\b([0-3]?[0-9][-\s/](?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*[-\s/](?:20)?[2-9][0-9])\b", re.IGNORECASE),
]
REFERENCE_PATTERNS = [
    re.compile(r"(?:UPI\s*Ref|IMPS\s*Ref|NEFT\s*Ref|Ref(?:\s*No)?|UPI|संदर्भ)[\s:.-]*([0-9]{9,14})\b", re.IGNORECASE),
    re.compile(r"\b([0-9]{12})\b"),
]
VPA_PATTERN = re.compile(r"\b([a-zA-Z0-9._-]+@[a-zA-Z]{2,15})\b")
MERCHANT_CONTEXT_PATTERNS = [
    re.compile(r"\bat\s+([A-Z][A-Za-z0-9\s.&'-]{1,25}?)\s+(?:on|for|via|ref|avl)\b"),
    re.compile(r";\s*([A-Z][A-Za-z0-9\s.&'-]{1,28}?)\s+credited\b"),
    re.compile(r"\b(?:IMPS\s+to|NEFT\s+to|NEFT\s+from|IMPS\s+from|UPI\s+to|UPI\s+from|towards|from|to)\s+([A-Z][A-Za-z0-9\s.&'-]{2,28}?)(?:[.\n;]|(?:\s+(?:on|Ref|UPI|Bal|via|VPA|Tuition|Course)))"),
    re.compile(r"Info:\s*(?:UPI|IMPS|NEFT)[-/]([A-Za-z0-9\s.&'-]{2,25}?)(?:[.\n;]|$)"),
    re.compile(r"\bfor\s+(?:your\s+)?([A-Z][A-Za-z0-9\s.&'-]{2,20}?)\s+payment\b"),
]

class FinancialEntityExtractor:
    def extract(self, text: str) -> Dict[str, Any]:
        raw = strip_instruction_prompt(text)
        lang, is_cm = detect_language_and_codemixing(raw)
        lower = raw.lower()

        # 1. Amount
        amount = None
        for pat in AMOUNT_PATTERNS:
            m = pat.search(raw)
            if m:
                amount = float(m.group(1).replace(",", ""))
                break

        # 2. Transaction Type
        d_pos = min([lower.find(k) for k in DEBIT_KEYWORDS if lower.find(k) != -1] or [999999])
        c_pos = min([lower.find(k) for k in CREDIT_KEYWORDS if lower.find(k) != -1] or [999999])
        txn_type = "debit" if d_pos <= c_pos else "credit"

        # 3. Bank
        bank = None
        best_bpos = 999999
        for b_canon, aliases in INDIAN_BANKS.items():
            for al in aliases:
                idx = lower.find(al)
                if idx != -1 and idx < best_bpos:
                    best_bpos = idx
                    bank = b_canon

        # 4. Account
        account = None
        for pat in ACCOUNT_PATTERNS:
            m = pat.search(raw)
            if m:
                account = m.group(1)
                break

        # 5. Date
        date_str = None
        for pat in DATE_PATTERNS:
            m = pat.search(raw)
            if m:
                date_str = m.group(1)
                break

        # 6. VPA & Reference
        vm = VPA_PATTERN.search(raw)
        vpa = vm.group(1) if vm else None
        reference = None
        for pat in REFERENCE_PATTERNS:
            rm = pat.search(raw)
            if rm and not rm.group(1).startswith("1800"):
                reference = rm.group(1)
                break

        # 7. Merchant / Beneficiary & Lexicon Hint
        merchant, beneficiary, lex_cat, is_p2m = None, None, None, False
        if vpa:
            vpa_clean = re.sub(r"[0-9_.-]+$", "", vpa.split("@")[0].lower())
            for k, (m_name, m_cat) in MERCHANT_LEXICON.items():
                if k.replace(" ", "") in vpa_clean:
                    merchant, lex_cat, is_p2m = m_name, m_cat, True
                    break

        if not merchant:
            for k, (m_name, m_cat) in sorted(MERCHANT_LEXICON.items(), key=lambda x: -len(x[0])):
                if re.search(rf"\b{re.escape(k)}\b", lower):
                    merchant, lex_cat, is_p2m = m_name, m_cat, True
                    break

        if not merchant:
            for pat in MERCHANT_CONTEXT_PATTERNS:
                cm = pat.search(raw)
                if cm:
                    cand = cm.group(1).strip(" .-;:")
                    if len(cand.split()) >= 2 and not any(b in cand.lower() for b in ["hotel", "direct", "mart", "bank", "university"]):
                        beneficiary, lex_cat, is_p2m = cand, "transfer", False
                    else:
                        merchant, is_p2m = cand, True
                    break

        return {
            "raw_sms": raw,
            "amount": amount,
            "type": txn_type,
            "bank": bank,
            "account": account,
            "date": date_str,
            "reference": reference,
            "vpa": vpa,
            "merchant": merchant,
            "beneficiary": beneficiary,
            "is_p2m": is_p2m,
            "lexicon_category_hint": lex_cat,
            "language": lang,
            "is_code_mixed": is_cm,
        }

extractor = FinancialEntityExtractor()
print("Sample Extraction Test:", extractor.extract("HDFC Bank: Rs.7,361.93 debited from A/c XX1489 on 03-11-25. VPA netmeds@icici. UPI Ref: 549264265572"))

## Step 4: Load `Ranjit0034/finee-dataset` from Hugging Face + Code-Mixed & Education Augmentation

In [ ]:
from datasets import load_dataset

TRAIN_LIMIT = 25000   # Set to 137267 to train on the entire 152K+ dataset!
EVAL_LIMIT  = 4000

print("Loading Ranjit0034/finee-dataset from Hugging Face...")
ds = load_dataset("Ranjit0034/finee-dataset", streaming=True)

def parse_hf_item(item: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    raw_sms, out = "", {}
    for m in item.get("messages", []):
        if m["role"] == "user":
            raw_sms = strip_instruction_prompt(m["content"])
        elif m["role"] == "assistant":
            try:
                out = json.loads(m["content"])
            except Exception:
                return None
    if not raw_sms or not out:
        return None
    cat = str(out.get("category", "shopping")).lower()
    if cat in {"cashback", "refund", ""}:
        cat = "shopping"
    lang, is_cm = detect_language_and_codemixing(raw_sms)
    lang = out.get("language", lang)
    return {
        "raw_sms": raw_sms,
        "normalized_text": normalize_financial_text(raw_sms),
        "amount": float(out["amount"]) if out.get("amount") is not None else None,
        "type": out.get("type", "debit"),
        "bank": out.get("bank"),
        "account": out.get("account"),
        "date": out.get("date"),
        "reference": out.get("reference"),
        "merchant": out.get("merchant"),
        "beneficiary": out.get("beneficiary"),
        "vpa": out.get("vpa"),
        "category": cat,
        "synopsis_category": SYNOPSIS_CATEGORY_MAP.get(cat, "Shopping"),
        "language": lang,
        "is_code_mixed": is_cm,
    }

def fetch_split(split_iter, limit: int) -> List[Dict[str, Any]]:
    rows = []
    for item in split_iter:
        p = parse_hf_item(item)
        if p:
            rows.append(p)
            if len(rows) >= limit:
                break
    return rows

train_rows = fetch_split(ds["train"], TRAIN_LIMIT)
val_key = "validation" if "validation" in ds else "valid"
valid_rows = fetch_split(ds[val_key], EVAL_LIMIT)
test_rows  = fetch_split(ds["test"], EVAL_LIMIT)

def generate_synopsis_augmentation(n_samples: int = 1500, seed: int = 42) -> List[Dict[str, Any]]:
    rng = random.Random(seed)
    banks = ["HDFC", "SBI", "ICICI", "Axis", "Kotak", "PNB", "BOB"]
    scenarios = [
        ("Swiggy", "swiggy@ybl", "food", 150, 950),
        ("Zomato", "zomato@hdfcbank", "food", 180, 1200),
        ("Zepto", "zepto@ybl", "grocery", 200, 1800),
        ("Amazon", "amazon@apl", "shopping", 399, 8500),
        ("Myntra", "myntra@ybl", "shopping", 599, 4500),
        ("Uber", "uber@axis", "transport", 90, 650),
        ("IRCTC", "irctc@sbi", "travel", 550, 3800),
        ("BESCOM", "bescom@ybl", "bills", 650, 3200),
        ("Apollo", "apollo@hdfcbank", "healthcare", 250, 2900),
        ("Udemy", "udemy@ybl", "education", 449, 3499),
        ("Coursera", "coursera@hdfcbank", "education", 1500, 4999),
        ("Unacademy", "unacademy@icici", "education", 2500, 18000),
        ("RV University", "rvuniversity@sbi", "education", 25000, 85000),
        ("Zerodha", "zerodha@hdfcbank", "investment", 2000, 45000),
    ]
    templates = [
        ("{bank} Alert: Aapke A/c XX{acct} se Rs.{amt} debit hua on {date} to {vpa} for {merch} payment. UPI Ref: {ref}", "hinglish", True),
        ("{bank}: A/c XX{acct} se INR {amt} kat gaya for {merch} payment on {date}. VPA {vpa}. Ref: {ref}", "hinglish", True),
        ("{bank} Alert: Nimma A/c XX{acct} inda Rs.{amt} {merch} ge UPI mulaka debit agide on {date}. VPA {vpa}. Ref: {ref}", "hinglish", True),
        ("{bank} Bank: Unga A/c XX{acct} la irundhu Rs.{amt} {merch} ku debit aagiduchu on {date}. VPA {vpa}. UPI Ref: {ref}", "hinglish", True),
        ("{bank} Bank: Rs.{amt} paid from A/c XX{acct} on {date} towards {merch} Course/Tuition Fee. VPA {vpa}. Ref: {ref}", "en", False),
    ]
    aug = []
    for _ in range(n_samples):
        merch, vpa, cat, lo, hi = rng.choice(scenarios)
        tpl, lang, is_cm = rng.choice(templates)
        bank = rng.choice(banks)
        acct = f"{rng.randint(1000, 9999)}"
        date = f"{rng.randint(1,28):02d}-{rng.randint(1,12):02d}-2025"
        ref = f"{rng.randint(100000000000, 999999999999)}"
        amt = round(rng.uniform(lo, hi), 2)
        sms = tpl.format(bank=bank, acct=acct, amt=f"{amt:.2f}", date=date, vpa=vpa, merch=merch, ref=ref)
        aug.append({
            "raw_sms": sms,
            "normalized_text": normalize_financial_text(sms),
            "amount": amt,
            "type": "debit",
            "bank": bank,
            "account": acct,
            "date": date,
            "reference": ref,
            "merchant": merch,
            "beneficiary": None,
            "vpa": vpa,
            "category": cat,
            "synopsis_category": SYNOPSIS_CATEGORY_MAP[cat],
            "language": lang,
            "is_code_mixed": is_cm,
        })
    return aug

df_train = pd.DataFrame(train_rows + generate_synopsis_augmentation(1600, seed=42)).sample(frac=1.0, random_state=42).reset_index(drop=True)
df_valid = pd.DataFrame(valid_rows + generate_synopsis_augmentation(350, seed=101)).sample(frac=1.0, random_state=42).reset_index(drop=True)
df_test  = pd.DataFrame(test_rows  + generate_synopsis_augmentation(350, seed=202)).sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f"Dataset Ready -> Train: {len(df_train):,} | Valid: {len(df_valid):,} | Test: {len(df_test):,}")
df_train.head(5)

## Step 5: Train Classical & Subword ML Models (Word + Character N-Gram TF-IDF with SVM, LogReg, LightGBM)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import lightgbm as lgb

vectorizer = FeatureUnion([
    ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), max_features=25000, sublinear_tf=True, min_df=2)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=30000, sublinear_tf=True, min_df=3)),
])

print("Extracting Word + Character Subword TF-IDF features...")
X_train = vectorizer.fit_transform(df_train["normalized_text"])
y_train = df_train["category"].values
X_test  = vectorizer.transform(df_test["normalized_text"])
y_test  = df_test["category"].values

print("1. Training Support Vector Machine (Calibrated LinearSVC)...")
svm_model = CalibratedClassifierCV(LinearSVC(C=1.0, class_weight="balanced", random_state=42), cv=3)
svm_model.fit(X_train, y_train)

print("2. Training Multinomial Logistic Regression...")
lr_model = LogisticRegression(C=2.0, class_weight="balanced", max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

print("3. Training LightGBM Gradient Boosted Trees...")
lgb_model = lgb.LGBMClassifier(n_estimators=120, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1)
lgb_model.fit(X_train, y_train)

for name, clf in [("SVM (LinearSVC + Char/Word TF-IDF)", svm_model), ("Logistic Regression", lr_model), ("LightGBM", lgb_model)]:
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    _, _, mf1, _ = precision_recall_fscore_support(y_test, preds, average="macro", zero_division=0)
    print(f"{name:38s} -> Test Accuracy: {acc*100:.2f}% | Macro F1: {mf1*100:.2f}%")

## Step 6: Fine-Tune Multilingual Transformer (`google/muril-base-cased` or `xlm-roberta-base`) on Colab GPU

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

TRANSFORMER_MODEL_NAME = "google/muril-base-cased"

unique_labels = sorted(df_train["category"].unique().tolist())
label2id = {l: i for i, l in enumerate(unique_labels)}
id2label = {i: l for l, i in label2id.items()}

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Fine-tuning {TRANSFORMER_MODEL_NAME} on {device}...")

tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL_NAME)
transformer_model = AutoModelForSequenceClassification.from_pretrained(
    TRANSFORMER_MODEL_NAME,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id,
).to(device)

class FinancialSMSDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(list(texts), truncation=True, padding=True, max_length=96, return_tensors="pt")
        self.labels = torch.tensor([label2id[y] for y in labels], dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item["labels"] = self.labels[idx]
        return item

df_tr_dl = df_train.sample(n=min(12000, len(df_train)), random_state=42)
train_loader = DataLoader(FinancialSMSDataset(df_tr_dl["normalized_text"], df_tr_dl["category"]), batch_size=32, shuffle=True)
test_loader  = DataLoader(FinancialSMSDataset(df_test["normalized_text"], df_test["category"]), batch_size=64, shuffle=False)

optimizer = torch.optim.AdamW(transformer_model.parameters(), lr=3e-5, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

for epoch in range(2):
    transformer_model.train()
    running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        inputs = {k: v.to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            outputs = transformer_model(**inputs)
            loss = outputs.loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/2 — Loss: {running_loss/len(train_loader):.4f}")

# Evaluate Transformer on Test Set
transformer_model.eval()
trans_preds, trans_probs = [], []
with torch.no_grad():
    for batch in test_loader:
        batch.pop("labels")
        inputs = {k: v.to(device) for k, v in batch.items()}
        logits = transformer_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        preds = np.argmax(probs, axis=-1)
        trans_probs.append(probs)
        trans_preds.extend([id2label[int(i)] for i in preds])

trans_probs = np.vstack(trans_probs)
t_acc = accuracy_score(y_test, trans_preds)
_, _, t_f1, _ = precision_recall_fscore_support(y_test, trans_preds, average="macro", zero_division=0)
print(f"Fine-Tuned {TRANSFORMER_MODEL_NAME} -> Test Accuracy: {t_acc*100:.2f}% | Macro F1: {t_f1*100:.2f}%")

## Step 7: Full Evaluation Suite (Model Comparison + Language Breakdown + Confusion Matrix)

In [ ]:
svm_probs = svm_model.predict_proba(X_test)
hybrid_preds = []
for idx, row in df_test.iterrows():
    ext = extractor.extract(row["raw_sms"])
    comb = 0.5 * svm_probs[idx] + 0.5 * trans_probs[idx]
    if ext["lexicon_category_hint"] in label2id:
        comb[label2id[ext["lexicon_category_hint"]]] += 0.65
    hybrid_preds.append(id2label[int(np.argmax(comb))])

summary_rows = []
for m_name, preds in [
    ("Logistic Regression (Subword TF-IDF)", lr_model.predict(X_test)),
    ("LightGBM (Gradient Boosting)", lgb_model.predict(X_test)),
    ("SVM (LinearSVC + Word/Char TF-IDF)", svm_model.predict(X_test)),
    (f"Multilingual Transformer ({TRANSFORMER_MODEL_NAME})", trans_preds),
    ("Hybrid Ensemble Framework (Ours)", hybrid_preds),
]:
    acc = accuracy_score(y_test, preds)
    mp, mr, mf1, _ = precision_recall_fscore_support(y_test, preds, average="macro", zero_division=0)
    _, _, wf1, _ = precision_recall_fscore_support(y_test, preds, average="weighted", zero_division=0)
    summary_rows.append({
        "Model": m_name,
        "Accuracy (%)": round(acc * 100, 2),
        "Macro Precision (%)": round(mp * 100, 2),
        "Macro Recall (%)": round(mr * 100, 2),
        "Macro F1 (%)": round(mf1 * 100, 2),
        "Weighted F1 (%)": round(wf1 * 100, 2),
    })

df_benchmark = pd.DataFrame(summary_rows)
print("=== OVERALL MODEL COMPARISON ===")
display(df_benchmark)

df_test_eval = df_test.copy()
df_test_eval["pred"] = hybrid_preds
lang_rows = []
for lang_code, grp in df_test_eval.groupby("language"):
    l_acc = accuracy_score(grp["category"], grp["pred"])
    _, _, l_f1, _ = precision_recall_fscore_support(grp["category"], grp["pred"], average="macro", zero_division=0)
    lang_rows.append({
        "Language Code": lang_code,
        "Language Name": SUPPORTED_LANGUAGES.get(lang_code, lang_code),
        "Test Samples": len(grp),
        "Accuracy (%)": round(l_acc * 100, 2),
        "Macro F1 (%)": round(l_f1 * 100, 2),
    })
print("\n=== MULTILINGUAL & CODE-MIXED PERFORMANCE ===")
display(pd.DataFrame(lang_rows))

cm = confusion_matrix(y_test, hybrid_preds, labels=unique_labels, normalize="true") * 100
plt.figure(figsize=(11, 8))
sns.heatmap(cm, annot=True, fmt=".1f", cmap="Blues", xticklabels=unique_labels, yticklabels=unique_labels)
plt.title("Row-Normalized Confusion Matrix (%) — Hybrid Financial NLP Framework", fontweight="bold")
plt.xlabel("Predicted Category")
plt.ylabel("True Category")
plt.tight_layout()
plt.show()

## Step 8: Export & Download Trained Models (`trained_models.zip`) to Your MacBook 💻
Run this cell to package your trained **Classical ML models (`classical_categorizers.pkl`)** and **Fine-Tuned MuRIL Transformer (`multilingual_transformer/`)** into `trained_models.zip` and download it directly to your Mac.

Once downloaded, unzip it into your local `/Users/prathikkb/Documents/NLP/` folder so that the `models/` directory contains both files.

In [ ]:
import pickle
import shutil
from pathlib import Path
from google.colab import files

export_dir = Path("models")
export_dir.mkdir(exist_ok=True)

print("1. Saving Classical ML Models (TF-IDF Vectorizer + SVM + LogReg + LightGBM)...")
with open(export_dir / "classical_categorizers.pkl", "wb") as f:
    pickle.dump(
        {
            "vectorizer": vectorizer,
            "models": {
                "svm": svm_model,
                "logreg": lr_model,
                "lightgbm": lgb_model,
            },
            "classes": unique_labels,
        },
        f,
    )

print("2. Saving Fine-Tuned MuRIL Transformer & Tokenizer...")
tr_dir = export_dir / "multilingual_transformer"
tr_dir.mkdir(exist_ok=True)
transformer_model.save_pretrained(tr_dir)
tokenizer.save_pretrained(tr_dir)

print("3. Creating trained_models.zip archive...")
zip_path = shutil.make_archive("trained_models", "zip", root_dir=".", base_dir="models")
print(f"✅ Archive created: {zip_path}. Triggering browser download...")
files.download(zip_path)